In [36]:
import pandas as pd
from pathlib import Path

In [37]:
processed_path = Path("../data/processed")

glucose = pd.read_csv(
    processed_path / "glucose_level.csv",
    parse_dates=["ts"]
)

meal = pd.read_csv(
    processed_path / "meal.csv",
    parse_dates=["ts"]
)

bolus = pd.read_csv(
    processed_path / "bolus.csv",
    parse_dates=["ts_begin"]
)

basal = pd.read_csv(
    processed_path / "basal.csv",
    parse_dates=["ts"]
)

temp_basal = pd.read_csv(
    processed_path / "temp_basal.csv",
    parse_dates=["ts_begin"]
)

exercise = pd.read_csv(
    processed_path / "exercise.csv",
    parse_dates=["ts"]
)

In [38]:
print(glucose.shape)
print(meal.shape)
print(bolus.shape)

(65535, 4)
(702, 5)
(1568, 6)


In [39]:
glucose = glucose.rename(columns = {
    'ts': 'timestamp',
    'value': 'glucose_value'
})

meal = meal.rename(columns = {
    'ts': 'timestamp',
    'carbs': 'meal_carbs',
    'type': 'meal_type'
})

bolus = bolus.rename(columns = {
    'ts_begin': 'timestamp',
    'dose': 'bolus_dose',
    'type': 'bolus_type'
})

basal = basal.rename(columns = {
    'ts': 'timestamp',
    'value': 'basal_value'
})

temp_basal = temp_basal.rename(columns = {
    'ts_begin': 'timestamp',
    'value': 'temp_basal_value'
})

exercise = exercise.rename(columns = {
    'ts': 'timestamp',
    'type': 'exercise_type',
    'duration': 'exercise_duration',
    'intensity': 'exercise_intensity'
})

In [40]:
datasets = [
    glucose,
    meal,
    bolus,
    basal,
    temp_basal,
    exercise
]

for df in datasets:
    df.sort_values(
        ['Patient', 'timestamp'],
        inplace=True
    )

In [41]:
merged = glucose.copy()

merged.head()

,Patient,Section,timestamp,glucose_value
43635,540-ws-training,glucose_level,2027-05-19 11:36:29,76
43638,540-ws-training,glucose_level,2027-05-19 11:41:29,72
43641,540-ws-training,glucose_level,2027-05-19 11:46:29,68
43644,540-ws-training,glucose_level,2027-05-19 11:51:29,65
43647,540-ws-training,glucose_level,2027-05-19 11:56:29,63


In [42]:
merged = pd.merge_asof(
    merged.sort_values("timestamp"),
    meal.sort_values("timestamp"),
    on="timestamp",
    by="Patient",
    direction="backward"
)

merged.head()

,Patient,Section_x,timestamp,glucose_value,Section_y,meal_type,meal_carbs
0,552-ws-training,glucose_level,2025-04-16 11:17:05,95,NaN,NaN,NaN
1,552-ws-training,glucose_level,2025-04-16 11:22:05,86,NaN,NaN,NaN
2,552-ws-training,glucose_level,2025-04-16 11:27:05,81,NaN,NaN,NaN
3,552-ws-training,glucose_level,2025-04-16 11:32:05,81,NaN,NaN,NaN
4,552-ws-training,glucose_level,2025-04-16 11:37:05,82,NaN,NaN,NaN


In [43]:
print(merged.shape)
print(merged.columns)

(65535, 7)
Index(['Patient', 'Section_x', 'timestamp', 'glucose_value', 'Section_y',
       'meal_type', 'meal_carbs'],
      dtype='object')


In [44]:
print(merged["meal_carbs"].notna().sum())

64026


In [45]:
merged[merged["meal_carbs"].notna()].head(10)

,Patient,Section_x,timestamp,glucose_value,Section_y,meal_type,meal_carbs
81,552-ws-training,glucose_level,2025-04-16 18:02:06,228,meal,Dinner,30.0
82,552-ws-training,glucose_level,2025-04-16 18:07:06,232,meal,Dinner,30.0
83,552-ws-training,glucose_level,2025-04-16 18:12:06,234,meal,Dinner,30.0
84,552-ws-training,glucose_level,2025-04-16 18:17:06,233,meal,Dinner,30.0
85,552-ws-training,glucose_level,2025-04-16 18:22:06,234,meal,Dinner,30.0
86,552-ws-training,glucose_level,2025-04-16 18:27:06,233,meal,Dinner,30.0
87,552-ws-training,glucose_level,2025-04-16 18:32:06,230,meal,Dinner,30.0
88,552-ws-training,glucose_level,2025-04-16 18:37:06,235,meal,Dinner,30.0
89,552-ws-training,glucose_level,2025-04-16 18:42:06,242,meal,Dinner,30.0
90,552-ws-training,glucose_level,2025-04-16 18:47:06,241,meal,Dinner,30.0


In [46]:
merged = merged.drop(
    columns=["Section_x", "Section_y"],
    errors="ignore"
)

In [47]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs']


In [48]:
print(bolus.columns.tolist())

['Patient', 'Section', 'timestamp', 'ts_end', 'bolus_type', 'bolus_dose']


In [49]:
bolus = bolus[[
    "Patient",
    "timestamp",
    "bolus_type",
    "bolus_dose"
]].copy()

In [50]:
# def merge_patient_asof(left_df, right_df):

#     merged_list = []

#     for patient in sorted(left_df["Patient"].unique()):

#         left = (
#             left_df[left_df["Patient"] == patient]
#             .sort_values("timestamp")
#             .reset_index(drop=True)
#         )

#         right = (
#             right_df[right_df["Patient"] == patient]
#             .sort_values("timestamp")
#             .reset_index(drop=True)
#         )

#         temp = pd.merge_asof(
#             left,
#             right,
#             on="timestamp",
#             direction="backward"
#         )

#         merged_list.append(temp)

#     return pd.concat(merged_list, ignore_index=True)

In [51]:
def merge_patient_asof(left_df, right_df):

    merged_list = []

    for patient in sorted(left_df["Patient"].unique()):

        left = (
            left_df[left_df["Patient"] == patient]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        right = (
            right_df[right_df["Patient"] == patient]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        # Remove Patient column from the right dataframe
        right = right.drop(columns=["Patient"], errors="ignore")

        temp = pd.merge_asof(
            left,
            right,
            on="timestamp",
            direction="backward"
        )

        merged_list.append(temp)

    return pd.concat(merged_list, ignore_index=True)

In [52]:
merged = merge_patient_asof(merged, bolus)

In [53]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose']


In [54]:
def merge_patient_asof(left_df, right_df):

    merged_list = []

    for patient in sorted(left_df["Patient"].unique()):

        left = (
            left_df[left_df["Patient"] == patient]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        right = (
            right_df[right_df["Patient"] == patient]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        # Remove Patient column from the right dataframe
        right = right.drop(columns=["Patient"], errors="ignore")

        temp = pd.merge_asof(
            left,
            right,
            on="timestamp",
            direction="backward"
        )

        merged_list.append(temp)

    return pd.concat(merged_list, ignore_index=True)

In [55]:
print(basal.columns.tolist())

['Patient', 'Section', 'timestamp', 'basal_value']


In [56]:
basal = basal[[
    'Patient',
    'timestamp',
    'basal_value'
]].copy()

In [57]:
print(basal.head())
print(basal.columns.tolist())

             Patient           timestamp  basal_value
228  540-ws-training 2027-05-19 00:00:00         0.80
229  540-ws-training 2027-05-19 05:00:00         1.05
230  540-ws-training 2027-05-19 09:00:00         0.95
231  540-ws-training 2027-05-19 14:00:00         0.40
232  540-ws-training 2027-05-22 00:00:00         0.80
['Patient', 'timestamp', 'basal_value']


In [58]:
merged = merge_patient_asof(merged, basal)

In [59]:
if "Patient_x" in merged.columns:
    merged = merged.rename(columns={"Patient_x": "Patient"})

if "Patient_y" in merged.columns:
    merged = merged.drop(columns=["Patient_y"])

In [60]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value']


In [61]:
merged.head(10)

,Patient,timestamp,glucose_value,meal_type,meal_carbs,bolus_type,bolus_dose,basal_value
0,540-ws-training,2027-05-19 11:36:29,76,NaN,NaN,normal,0.8,0.95
1,540-ws-training,2027-05-19 11:41:29,72,NaN,NaN,normal,0.8,0.95
2,540-ws-training,2027-05-19 11:46:29,68,NaN,NaN,normal,0.8,0.95
3,540-ws-training,2027-05-19 11:51:29,65,NaN,NaN,normal,0.8,0.95
4,540-ws-training,2027-05-19 11:56:29,63,NaN,NaN,normal,0.8,0.95
5,540-ws-training,2027-05-19 12:01:29,66,NaN,NaN,normal,0.8,0.95
6,540-ws-training,2027-05-19 12:06:29,71,NaN,NaN,normal,0.8,0.95
7,540-ws-training,2027-05-19 12:11:29,78,NaN,NaN,normal dual,5.5,0.95
8,540-ws-training,2027-05-19 12:16:29,90,NaN,NaN,normal dual,5.5,0.95
9,540-ws-training,2027-05-19 12:21:29,99,NaN,NaN,normal dual,5.5,0.95


In [62]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value']


In [63]:
temp_basal = temp_basal[[
    'Patient',
    'timestamp',
    'temp_basal_value'
]].copy()

In [64]:
print(temp_basal.columns.tolist())

['Patient', 'timestamp', 'temp_basal_value']


In [65]:
merged = merge_patient_asof(merged, temp_basal)

In [66]:
if "Patient_x" in merged.columns:
    merged = merged.rename(columns={"Patient_x": "Patient"})

if "Patient_y" in merged.columns:
    merged = merged.drop(columns=["Patient_y"])

In [67]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value']


In [68]:
exercise = exercise[[
    'Patient',
    'timestamp',
    'exercise_type',
    'exercise_duration',
    'exercise_intensity'
]].copy()

In [69]:
merged = merge_patient_asof(merged, exercise)

In [70]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity']


In [72]:
finger_stick_path = processed_path / "finger_stick.csv"

if not finger_stick_path.exists():
    raise FileNotFoundError(f"{finger_stick_path} not found. Load the finger_stick dataset first.")

finger_stick = pd.read_csv(
    finger_stick_path,
    parse_dates=["ts"]
).rename(columns={"ts": "timestamp"})

finger_stick = finger_stick[[
    "Patient",
    "timestamp",
    "value"
]].copy()

In [73]:
merged = merge_patient_asof(merged, finger_stick)

In [74]:
print(merged.columns.tolist())

['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'value']


In [ ]:
basic_gsr_path = processed_path / "basic_gsr.csv"

if not basic_gsr_path.exists():
    basic_gsr_path = processed_path / "basic_gsr.csv"

    if basic_gsr_path.exists():
        basic_gsr = pd.read_csv(
            basic_gsr_path,
            parse_dates=["ts"]
        ).rename(columns={"ts": "timestamp"})

        basic_gsr = basic_gsr[[
            "Patient",
            "timestamp",
            "value"
        ]].copy()
    else:
        basic_gsr = pd.DataFrame(columns=["Patient", "timestamp", "value"])
        print(f"Warning: {basic_gsr_path} not found. basic_gsr will be empty.")

basic_gsr = pd.read_csv(
    basic_gsr_path, 
    parse_dates=["ts"]
).rename(columns={"ts": "timestamp"})

basic_gsr = basic_gsr[[
    "Patient",
    "timestamp",
    "value"
]].copy()



FileNotFoundError: ..\data\processed\basic_gsr.csv not found. Load the basic_gsr dataset first.